# Day 8: Snowflake Storage Integration & Dynamic Tables (Mock)

This notebook uses DuckDB to simulate Snowflake's incremental query behavior and SQL transformation logic.

In [ ]:
!pip install duckdb

In [ ]:
import duckdb

conn = duckdb.connect(':memory:')
print("DuckDB in-memory database connected.")

In [ ]:
# Simulate Silver Table (Iceberg equivalent)
conn.execute("""
CREATE TABLE silver_condition_occurrence (
    condition_occurrence_id INT,
    person_id VARCHAR,
    condition_concept_id INT,
    condition_start_date DATE
);
""")

conn.execute("""
INSERT INTO silver_condition_occurrence VALUES 
(1, 'P1', 32020, '2023-01-01'),
(2, 'P2', 32020, '2023-01-05'),
(3, 'P1', 43210, '2023-02-01');
""")

print("Silver table created and populated.")
print(conn.execute("SELECT * FROM silver_condition_occurrence").df())

In [ ]:
# Simulate Dynamic Table logic (Materialized View in DuckDB)
# Snowflake Dynamic Tables update continuously based on target_lag.
# Here we simulate the SQL logic that would run inside it.
conn.execute("""
CREATE VIEW gold_condition_rollup_dt AS
SELECT 
    condition_concept_id,
    COUNT(condition_occurrence_id) as condition_count,
    MAX(condition_start_date) as last_seen_date
FROM silver_condition_occurrence
GROUP BY condition_concept_id;
""")

print("Gold Dynamic Table (View) query output:")
print(conn.execute("SELECT * FROM gold_condition_rollup_dt").df())

In [ ]:
# Simulate new data arriving in Silver
conn.execute("""
INSERT INTO silver_condition_occurrence VALUES 
(4, 'P3', 32020, '2023-03-01');
""")

print("Gold Dynamic Table after new Silver data arrives:")
print(conn.execute("SELECT * FROM gold_condition_rollup_dt").df())